In [1]:
from pathlib import Path
import sys, json, hashlib, importlib.util, zipfile
import pandas as pd
ROOT=Path("/home/featurize/work/biological_prior_validation").resolve()
if not ROOT.is_dir():
    raise FileNotFoundError(f"Please change ROOT to the existing project directory：{ROOT}")


In [2]:
SOURCE = '"""Task 2.1: immutable method/control rules, no real-data fitting or target access."""\nfrom pathlib import Path\nimport hashlib,json,sys,subprocess,platform\nimport numpy as np\nimport pandas as pd\nimport scipy\nimport scipy.sparse as sp\nfrom scipy.sparse.linalg import eigsh\nENGINE_SHA=hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\nLINES=[\'A172\',\'T98G\',\'U87MG\'];DRUGS=[\'Vehicle\',\'Lapatinib\',\'Nintedanib\',\'Trametinib\',\'ZSTK474\'];REPS=[\'replicate_1\',\'replicate_2\']\nFOUR=[(\'A172\',\'Lapatinib\'),(\'A172\',\'Trametinib\'),(\'T98G\',\'Nintedanib\'),(\'U87MG\',\'ZSTK474\')]\n\ndef digest(p):\n h=hashlib.sha256()\n with Path(p).open(\'rb\') as f:\n  for b in iter(lambda:f.read(1048576),b\'\'):h.update(b)\n return h.hexdigest()\ndef js(p):return json.loads(Path(p).read_text())\ndef save(p,obj):\n p=Path(p);p.parent.mkdir(parents=True,exist_ok=True);temp=p.with_name(p.name+\'.tmp\');temp.write_text(json.dumps(obj,ensure_ascii=False,indent=2,allow_nan=False,default=str));temp.replace(p)\ndef normalized(obj):return json.loads(json.dumps(obj,allow_nan=False,default=str))\ndef freeze(p,obj):\n obj=normalized(obj)\n if Path(p).exists() and js(p)!=obj:raise ValueError(f\'Frozen contract changed: {p}; do not overwrite after target exposure\')\n save(p,obj)\ndef default_config(root):\n root=Path(root)\n return {\'project_root\':str(root),\'run_path\':str(root/\'data/splits/step2_q05_seed20261008_v1\'),\n \'development_name\':\'inner_q05_seed20261008_v1\',\'prior_experiment\':\'step5_four_contexts_v1\',\'protocol_name\':\'task2_five_methods_v1\',\'master_seed\':20261008,\n \'alpha_coarse\':[0.,.001,.01,.1,1.,10.,100.],\n \'beta_coarse\':[0.,1e-6,1e-5,1e-4,.001,.01,.1,1.,10.,100.],\n \'lambda_threshold_quantiles\':[.25,.5,.75,.9,.97],\'lambda_high_multiplier\':3.,\n \'initial_random_seeds\':20,\'expanded_random_seeds\':100,\'swap_multiplier\':10,\'max_attempts_per_swap\':100,\n \'cg_rtol\':1e-8,\'cg_maxiter\':1000,\'adaptive_max_outer\':100,\'adaptive_x_rtol\':1e-7,\'adaptive_s_atol\':1e-6}\n\ndef validate_config(c):\n for k in [\'protocol_name\',\'development_name\',\'prior_experiment\']:\n  if not c[k] or Path(c[k]).name!=c[k] or c[k] in [\'.\',\'..\']:raise ValueError(f\'Invalid {k}\')\n for k in [\'alpha_coarse\',\'beta_coarse\']:\n  v=np.asarray(c[k],float)\n  if v.ndim!=1 or not len(v) or not np.isfinite(v).all() or (v<0).any() or not np.array_equal(v,np.unique(v)) or 0 not in v:raise ValueError(f\'Invalid {k}; sorted unique nonnegative values including zero required\')\n if c[\'initial_random_seeds\']!=20 or c[\'expanded_random_seeds\']!=100:raise ValueError(\'PDF initial/expanded seed counts are 20/100\')\n for k in [\'master_seed\',\'swap_multiplier\',\'max_attempts_per_swap\',\'cg_maxiter\',\'adaptive_max_outer\']:\n  if not isinstance(c[k],int) or c[k]<(0 if k==\'master_seed\' else 1):raise ValueError(f\'Invalid {k}\')\n for k in [\'cg_rtol\',\'adaptive_x_rtol\',\'adaptive_s_atol\',\'lambda_high_multiplier\']:\n  if not np.isfinite(c[k]) or c[k]<=0:raise ValueError(f\'Invalid {k}\')\n q=np.asarray(c[\'lambda_threshold_quantiles\'],float)\n if not len(q) or not np.isfinite(q).all() or not ((q>0)&(q<1)).all() or not np.array_equal(q,np.unique(q)):raise ValueError(\'Invalid threshold quantiles\')\n\ndef canonical_edges(edges,n_genes,weights=None):\n raw=np.asarray(edges)\n if raw.ndim!=2 or raw.shape[1]!=2 or raw.dtype.kind not in \'iu\' or not isinstance(n_genes,int) or n_genes<1:raise ValueError(\'Integer edge pairs and positive node count required\')\n e=raw.astype(np.int64,copy=True);e.sort(axis=1)\n if (e<0).any() or (e>=n_genes).any() or (e[:,0]==e[:,1]).any():raise ValueError(\'Invalid endpoint or self loop\')\n order=np.lexsort((e[:,1],e[:,0]));e=e[order]\n if len(e) and len(np.unique(e,axis=0))!=len(e):raise ValueError(\'Duplicate undirected edges\')\n if weights is None:return e\n w=np.asarray(weights,float)\n if w.shape!=(len(e),) or not np.isfinite(w).all() or (w<0).any():raise ValueError(\'Invalid edge weights\')\n return e,w[order]\n\ndef graph_laplacian(edges,weights,n_genes):\n e,w=canonical_edges(edges,n_genes,weights)\n W=sp.csr_matrix((np.r_[w,w],(np.r_[e[:,0],e[:,1]],np.r_[e[:,1],e[:,0]])),shape=(n_genes,n_genes))\n return (sp.diags(np.asarray(W.sum(axis=1)).ravel())-W).tocsr()\n\ndef graph_audit(edges,weights,n_genes):\n e,w=canonical_edges(edges,n_genes,weights);L=graph_laplacian(e,w,n_genes)\n degree=np.bincount(e.ravel(),minlength=n_genes);weighted=np.bincount(e[:,0],weights=w,minlength=n_genes)+np.bincount(e[:,1],weights=w,minlength=n_genes)\n if n_genes<=256:maximum=float(np.linalg.eigvalsh(L.toarray()).max())\n elif not np.any(w):maximum=0.\n else:maximum=float(eigsh(L,k=1,which=\'LA\',return_eigenvectors=False,v0=np.random.Generator(np.random.PCG64(0)).standard_normal(n_genes),tol=1e-10)[0])\n return {\'support_edges\':len(e),\'positive_weight_edges\':int((w>0).sum()),\'zero_weight_edges\':int((w==0).sum()),\'weight_sum\':float(w.sum()),\'trace_Lg\':float(L.diagonal().sum()),\'lambda_max\':max(maximum,0.),\'degree_sequence\':degree.tolist(),\'weighted_degree\':weighted.tolist()}\n\ndef normalize_base(edges,weights,n_genes):\n e,w=canonical_edges(edges,n_genes,weights);before=graph_audit(e,w,n_genes);scale=before[\'lambda_max\']\n if scale<=0:raise ValueError(\'Cannot spectral-normalize a zero base graph\')\n after=graph_audit(e,w/scale,n_genes)\n return e,w/scale,{\'normalization\':\'divide by this base graph spectral radius; reuse fixed scale for its derived graphs\',\'scale\':scale,\'before\':before,\'after\':after}\n\ndef rng_for(master,method,scope,seed_index,phase):\n ids={\'M3\':3,\'M4\':4};phases={\'inner\':0,\'outer\':1}\n for value in [master,scope,seed_index]:\n  if not isinstance(value,(int,np.integer)) or value<0:raise ValueError(\'Invalid deterministic stream ID\')\n if method not in ids or phase not in phases:raise ValueError(\'Invalid stream method/phase\')\n return np.random.Generator(np.random.PCG64(np.random.SeedSequence([int(master),ids[method],int(scope),int(seed_index),phases[phase]])))\n\ndef matched_permutation(edges,effective_weights,n_genes,rng):\n e,w=canonical_edges(edges,n_genes,effective_weights);permutation=rng.permutation(len(w));new=w[permutation]\n # These are effective w0*s, INCLUDING zeros, not s values alone.\n before=graph_audit(e,w,n_genes);after=graph_audit(e,new,n_genes)\n if not np.array_equal(np.sort(new),np.sort(w)) or before[\'zero_weight_edges\']!=after[\'zero_weight_edges\'] or not np.isclose(before[\'trace_Lg\'],after[\'trace_Lg\'],rtol=1e-13,atol=1e-13):raise AssertionError(\'M3 invariants violated\')\n return e,new,{\'permutation\':permutation.tolist(),\'before\':before,\'after\':after,\'renormalized\':False,\'weighted_degree_preserved\':bool(np.allclose(before[\'weighted_degree\'],after[\'weighted_degree\']))}\n\ndef double_edge_swaps(edges,n_genes,rng,requested,max_attempts):\n e=canonical_edges(edges,n_genes);m=len(e)\n if not isinstance(requested,int) or requested<1 or not isinstance(max_attempts,int) or max_attempts<requested or m<2:raise ValueError(\'Invalid swap budget or insufficient edges\')\n pairs=[tuple(x) for x in e];support=set(pairs);accepted=0;attempts=0\n while accepted<requested and attempts<max_attempts:\n  attempts+=1;i,j=rng.choice(m,2,replace=False);a,b=pairs[i];c,d=pairs[j]\n  if len({a,b,c,d})!=4:continue\n  if rng.integers(2):c,d=d,c\n  x=tuple(sorted((a,c)));y=tuple(sorted((b,d)))\n  if x==y or x in support or y in support:continue\n  support.remove(pairs[i]);support.remove(pairs[j]);support.add(x);support.add(y);pairs[i]=x;pairs[j]=y;accepted+=1\n if accepted!=requested:raise RuntimeError(f\'M4 swaps unavailable/insufficient: accepted={accepted}, requested={requested}, attempts={attempts}; no unrewired graph may be treated as a successful null\')\n new=canonical_edges(np.asarray(pairs,dtype=int),n_genes)\n if not np.array_equal(np.bincount(e.ravel(),minlength=n_genes),np.bincount(new.ravel(),minlength=n_genes)):raise AssertionError(\'Unweighted degree sequence changed\')\n return new,{\'requested_swaps\':requested,\'accepted_swaps\':accepted,\'attempts\':attempts,\'edge_overlap_fraction\':len(set(map(tuple,e))&set(map(tuple,new)))/m,\'connectivity_preserved\':False,\'mixing_verified\':False,\'sampling_claim\':\'finite swap chain; not proof of uniform random graph sampling\'}\n\ndef rewired_control(edges,weights,n_genes,rng,swap_multiplier=10,max_attempts_per_swap=100):\n e,w=canonical_edges(edges,n_genes,weights)\n if not isinstance(swap_multiplier,int) or swap_multiplier<1 or not isinstance(max_attempts_per_swap,int) or max_attempts_per_swap<1:raise ValueError(\'Invalid M4 swap policy\')\n requested=swap_multiplier*len(e);new,swaps=double_edge_swaps(e,n_genes,rng,requested,requested*max_attempts_per_swap)\n perm=rng.permutation(len(w));raw_weights=w[perm]\n raw_old=graph_audit(e,w,n_genes);raw_new=graph_audit(new,raw_weights,n_genes)\n assert np.array_equal(raw_old[\'degree_sequence\'],raw_new[\'degree_sequence\'])\n assert np.array_equal(np.sort(w),np.sort(raw_weights))\n new,scaled,norm=normalize_base(new,raw_weights,n_genes)\n return new,scaled,{\'swaps\':swaps,\'weight_assignment\':\'uniform permutation of original raw weights onto lexicographically ordered rewired pairs; no degree-based rematching\',\n \'raw_weight_permutation\':perm.tolist(),\'raw_reference\':raw_old,\'raw_rewired\':raw_new,\'normalization\':norm,\n \'weighted_degree_preserved\':bool(np.allclose(raw_old[\'weighted_degree\'],raw_new[\'weighted_degree\'])),\'weighted_degree_constraint_enforced\':False}\n\ndef conditional_edf(cell_eigenvalues,gene_laplacian,alpha,beta):\n """tr(H) holding representations, cell graph and adaptive edge weights fixed."""\n lc=np.asarray(cell_eigenvalues,float);L=gene_laplacian.toarray() if sp.issparse(gene_laplacian) else np.asarray(gene_laplacian,float)\n if lc.ndim!=1 or not len(lc) or not np.isfinite(lc).all() or (lc<-1e-9).any() or L.ndim!=2 or L.shape[0]<1 or L.shape[0]!=L.shape[1] or not np.isfinite(L).all() or not np.allclose(L,L.T,atol=1e-10):raise ValueError(\'Invalid conditional smoother spectrum\')\n if not np.isfinite(alpha) or not np.isfinite(beta) or alpha<0 or beta<0:raise ValueError(\'Invalid smoother parameters\')\n eig=np.linalg.eigvalsh(L)\n if eig.min()<-1e-8:raise ValueError(\'Gene Laplacian not PSD\')\n eig=np.maximum(eig,0);lc=np.maximum(lc,0)\n return float(np.sum(1/(1+alpha*lc[:,None]+beta*eig[None,:])))\n\ndef lambda_from_threshold(beta,tau):\n if not np.isfinite(beta) or beta<0 or not np.isfinite(tau) or tau<=0:raise ValueError(\'Finite positive tau and nonnegative beta required\')\n if beta==0:return 1. # inactive gene term; positive lambda fixes s=1\n return float(beta*tau/2)\n\ndef method_registry():\n return [\n {\'method\':\'M0\',\'name\':\'Cell-only\',\'optimization\':\'alpha only; beta=0\',\'graph\':\'none\',\'target_role\':\'development selection only\'},\n {\'method\':\'M1\',\'name\':\'Fixed PPI\',\'optimization\':\'joint Cartesian alpha x beta, including beta=0\',\'graph\':\'original STRING base graph\'},\n {\'method\':\'M2\',\'name\':\'Adaptive PPI\',\'optimization\':\'joint alpha x beta x lambda-threshold-rule; includes fixed-PPI infinity limit and beta=0\',\'graph\':\'w_effective=w0_scaled*s; s=clip(1-beta*d2/(2*lambda),0,1)\',\'new_fit_required\':True},\n {\'method\':\'M3\',\'name\':\'Matched random weighting\',\'optimization\':\'inherit selected M2 alpha/beta/lambda; no per-permutation tuning\',\'graph\':\'permute ALL effective M2 edge weights, including zeros, over original support; never renormalize\'},\n {\'method\':\'M4\',\'name\':\'Randomized PPI\',\'optimization\':\'independently joint alpha x beta per seed, same grid/rule/budget as M1\',\'graph\':\'simple undirected degree-preserving double-edge swaps; permute original raw weights; normalize each randomized BASE graph\'}]\n\ndef grid_contract(c):\n return {\'status\':\'coarse_rules_frozen_in_2.1; refinement_to_be_defined_and_frozen_in_2.2_before_benchmark\',\n \'alpha\':c[\'alpha_coarse\'],\'beta\':c[\'beta_coarse\'],\'M1_M4\':\'identical Cartesian grid, candidate count and development selection criterion; beta=0 required\',\n \'M2\':\'same alpha/beta Cartesian grid, additionally lambda rules for each positive beta; beta=0 is a single nonadaptive candidate per alpha\',\n \'lambda_rules\':{\'equation\':\'s=clip(1-beta*d2/(2*lambda),0,1); tau=2*lambda/beta for beta>0\',\'finite_thresholds\':\'quantiles of positive training-only cell-only edge energies for each alpha; high multiplier times q99\',\'quantiles\':c[\'lambda_threshold_quantiles\'],\'high_multiplier\':c[\'lambda_high_multiplier\'],\'lambda_value\':\'lambda=beta*tau/2; save both actual lambda and tau per candidate\',\'fixed_limit\':\'explicit lambda/tau infinity sentinel, not JSON numeric infinity\',\'zero_beta\':\'s=1, lambda=1 inactive, no redundant lambda search\',\'degenerate_training_energy\':\'use fixed/infinity only; report no positive energy; no target rescue\'},\n \'criterion\':\'exact mean per-cell development Poisson deviance, all common development pairs incl zeros; no final B selection\',\n \'tie_break\':\'deviance, alpha, beta; adaptive ties prefer fixed/infinity; then predefined lambda-rule order\',\n \'convergence\':{\'cg_rtol\':c[\'cg_rtol\'],\'cg_maxiter\':c[\'cg_maxiter\'],\'adaptive_max_outer\':c[\'adaptive_max_outer\'],\'adaptive_x_rtol\':c[\'adaptive_x_rtol\'],\'adaptive_s_atol\':c[\'adaptive_s_atol\'],\'failed_candidates\':\'keep full records/status; never fake valid scores\',\'precision_check\':\'fixed inputs/parameters; compare standard and tighter solvers using development only before benchmark; no B-driven precision/parameter selection\'},\n \'normalization\':{\'cell\':\'same SVD/cosine-kNN/metacell Galerkin rule for all methods, normalized cell spectral radius; fit from T for development, full A for final\',\'base_gene\':\'M1 and each M4 BASE graph use own positive spectral radius; record raw/scaled degrees, total weights, lambda_max\',\'derived_gene\':\'M2/M3 inherit original M1 base scale; no separate renormalization, preserving M3 effective weight multiset/trace\',\'M3_shrinkage_check\':\'record tr(Lg) and conditional tr(H); do not assume matched trace implies matched shrinkage; no final-B EDF matching\'},\n \'fine_grid\':\'NOT fixed here; Step 2.2 must freeze coarse-to-fine schedule and identical M1/M4 budget before any real benchmark\'}\n\ndef adaptive_reliability(X,edges,beta,lambda_value):\n X=np.asarray(X,float)\n if X.ndim!=2 or not np.isfinite(X).all() or not np.isfinite(beta) or beta<0:raise ValueError(\'Invalid adaptive inputs\')\n e=canonical_edges(edges,X.shape[1]);s=np.ones(len(e))\n if lambda_value==\'fixed_limit\':return s\n value=float(lambda_value)\n if not np.isfinite(value) or value<=0:raise ValueError(\'Positive finite lambda or fixed_limit required\')\n if beta==0:return s\n for lo in range(0,len(e),2048):\n  part=e[lo:lo+2048];diff=X[:,part[:,0]]-X[:,part[:,1]];d2=np.einsum(\'ij,ij->j\',diff,diff)\n  s[lo:lo+len(part)]=np.clip(1-beta*d2/(2*value),0,1)\n return s\n\ndef seed_plan(c):\n rows=[]\n for li,line in enumerate(LINES):\n  for di,drug in enumerate(DRUGS):\n   for ri,rep in enumerate(REPS):\n    scope=(li*len(DRUGS)+di)*len(REPS)+ri\n    for method in [\'M3\',\'M4\']:\n     for phase,phase_id in [(\'inner\',0),(\'outer\',1)]:\n      for index in range(c[\'expanded_random_seeds\']):\n       entropy=[c[\'master_seed\'],int(method[1]),scope,index,phase_id]\n       rows.append({\'scenario\':\'A\',\'scope_id\':scope,\'cell_line\':line,\'drug\':drug,\'replicate\':rep,\'method\':method,\'phase\':phase,\'seed_index\':index,\n       \'initial_round\':index<c[\'initial_random_seeds\'],\'context_round\':\'first_four_exploratory\' if (line,drug) in FOUR else \'later_all_contexts\',\n       \'rng\':\'PCG64(SeedSequence(entropy))\',\'entropy_json\':json.dumps(entropy),\'execution_status\':\'PLANNED_NOT_RUN\'})\n return pd.DataFrame(rows)\n\ndef write_environment(root,out):\n try:\n  res=subprocess.run([\'git\',\'-C\',str(root),\'rev-parse\',\'HEAD\'],text=True,capture_output=True,timeout=15)\n  git={\'commit\':res.stdout.strip() if res.returncode==0 else None,\'status\':\'available\' if res.returncode==0 else \'not_a_git_checkout\'}\n  if res.returncode==0:git[\'dirty\']=bool(subprocess.run([\'git\',\'-C\',str(root),\'status\',\'--porcelain\'],text=True,capture_output=True,timeout=15).stdout.strip())\n except (FileNotFoundError,subprocess.TimeoutExpired):git={\'commit\':None,\'status\':\'git_unavailable\'}\n save(out/\'environment.json\',{\'python\':sys.version,\'numpy\':np.__version__,\'pandas\':pd.__version__,\'scipy\':scipy.__version__,\'platform\':platform.platform(),\'git\':git,\'engine_sha256\':ENGINE_SHA})\n res=subprocess.run([sys.executable,\'-m\',\'pip\',\'freeze\'],capture_output=True,text=True,timeout=60)\n if res.returncode:raise RuntimeError(\'pip freeze failed\')\n (out/\'python_requirements.lock.txt\').write_text(res.stdout)\n\ndef prepare_contract(c):\n validate_config(c);root=Path(c[\'project_root\']).resolve();run=Path(c[\'run_path\']).resolve();dev=run/\'development\'/c[\'development_name\']\n if not run.is_relative_to(root/\'data/splits\'):raise ValueError(\'Run path must be inside project data/splits\')\n pairpath=run/\'A_preprocessing/evaluation_pair_contract.json\';metricpath=run/\'evaluation/metric_contract.json\';pair=js(pairpath);metric=js(metricpath)\n if metric[\'pair_contract_sha256\']!=digest(pairpath):raise ValueError(\'Step 3/2 contract mismatch\')\n cp=run/\'A_preprocessing\'/pair[\'cells_file\'];gp=run/\'A_preprocessing\'/pair[\'genes_file\']\n for p,key in [(cp,\'cells_sha256\'),(gp,\'genes_sha256\')]:\n  if digest(p)!=pair[key] or metric[key]!=pair[key]:raise ValueError(\'Frozen cell/gene files changed\')\n cells=pd.read_parquet(cp).sort_values(\'matrix_row\').reset_index(drop=True);genes=pd.read_parquet(gp).sort_values(\'selected_col\')\n if not np.array_equal(cells.matrix_row,np.arange(len(cells))) or not cells.cell_barcode.is_unique or not cells.evaluation_eligible.isin([True,False]).all():raise ValueError(\'Frozen cell mapping invalid\')\n if not np.array_equal(genes.selected_col,np.arange(len(genes))) or not genes.gene_idx.is_unique or not genes.gene_symbol.is_unique:raise ValueError(\'Frozen gene mapping invalid\')\n n=int(cells.evaluation_eligible.sum())\n for contract in [pair,metric]:\n  if (contract[\'n_cells\'],contract[\'n_genes\'],contract[\'n_pairs\'])!=(n,len(genes),n*len(genes)):raise ValueError(\'Frozen I size mismatch\')\n for col,values in [(\'cell_line\',LINES),(\'drug\',DRUGS),(\'replicate\',REPS)]:\n  if not cells.loc[cells.evaluation_eligible,col].isin(values).all():raise ValueError(\'Unknown eligible metadata\')\n if metric[\'q\']!=pair[\'q\'] or metric[\'normalization_scale\']!=pair[\'normalization_scale\']:raise ValueError(\'Count/normalization mapping changed\')\n dsum=js(dev/\'step4_summary.json\');dc=js(dev/\'scenario_A_contract.json\')\n if dsum[\'status\']!=\'step4_prepared\' or dsum[\'metric_contract_sha256_after\']!=digest(metricpath):raise ValueError(\'Step 4 incomplete or metric changed\')\n inner={}\n for rep in REPS:\n  p=dev/f\'{rep}_inner_selected_genes.parquet\';expected=next(x[\'selected_genes_sha256\'] for x in dsum[\'inner_genes_by_replicate\'] if x[\'replicate\']==rep)\n  if digest(p)!=expected:raise ValueError(\'Inner gene hash changed\')\n  inner[rep]={\'path\':str(p),\'sha256\':expected,\'scope\':\'T only, fixed same universe for M0-M4 within this phase/label\'}\n prior=run/\'step5\'/c[\'prior_experiment\'];previous=js(prior/\'step5_summary.json\');lockpath=run/\'evaluation/target_access_lock.json\'\n if previous[\'status\']!=\'four_context_baseline_scored\' or previous[\'final_B_accessed\'] is not True or not lockpath.is_file():raise ValueError(\'Expected audited Step 5 target exposure record\')\n history={\'prior_summary\':str(prior/\'step5_summary.json\'),\'prior_summary_sha256\':digest(prior/\'step5_summary.json\'),\'prior_registry_sha256\':digest(prior/\'frozen_model_registry.json\'),\n \'target_access_lock_sha256\':digest(lockpath),\'already_scored_contexts\':previous[\'contexts\'],\'already_accessed_final_B\':True,\n \'interpretation\':\'Task2 reuses exploratory molecular targets; not new confirmatory evaluation; no retrospective claim that target outcomes were unseen\',\n \'protection\':\'do not delete/replace Step5 lock or modify Step5 predictions; Task2 future registration must use distinct protocol/experiment identity and explicit reuse provenance\'}\n references={\'identity_sha256\':digest(run/\'identity.json\'),\'pair_sha256\':digest(pairpath),\'metric_sha256\':digest(metricpath),\'step4_identity_sha256\':digest(dev/\'identity.json\'),\'roles_sha256\':digest(dev/\'cell_roles.parquet\'),\n \'outer_gene_sha256\':digest(gp),\'inner_genes\':inner,\'ppi_reference_sha256\':digest(root/\'data/processed/ppi_annotation_reference_edges.parquet\')}\n out=run/\'task2_protocols\'/c[\'protocol_name\'];out.mkdir(parents=True,exist_ok=True)\n identity={\'version\':\'task2-step1-method-rules-v1\',\'engine_sha256\':ENGINE_SHA,\'config\':c,\'references\':references,\'target_history\':history,\'numpy\':np.__version__,\'scipy\':scipy.__version__}\n freeze(out/\'protocol_identity.json\',identity)\n configdir=root/\'configs\'/c[\'protocol_name\'];configdir.mkdir(parents=True,exist_ok=True)\n methods=method_registry();grids=grid_contract(c)\n common={\'scenario\':\'A\',\'development_input\':\'T from Step4 nested split; normalization/graphs/metacells/adaptive weights from T only\',\'development_target\':\'V for hyperparameter selection; same per-cell V I for all methods in phase\',\n \'scoring\':{\'outer_q\':metric[\'q\'],\'inner_q\':dc[\'inner_q\'],\'normalization_scale\':metric[\'normalization_scale\'],\'mu_floor\':metric[\'mu_floor\'],\'relative_min_denominator\':metric[\'relative_min_denominator\'],\'primary\':metric[\'primary\'],\'secondary\':metric[\'secondary\']},\n \'final_input\':\'refit on full A under frozen rules\',\'final_target\':\'B final evaluation only, no parameter selection/graph construction/context filtering\',\'outer_pair_contract\':str(pairpath),\'metric_contract\':str(metricpath),\n \'inner_gene_universe\':inner,\'outer_gene_universe\':{\'path\':str(gp),\'sha256\':digest(gp)},\'feature_scope\':\'inner train-selected and outer full-A universes may differ; no method-specific feature selection within a phase\',\n \'Lc\':\'one shared cached representation/operator per context/label/phase; all five methods use the exact same IDs/hash\',\n \'contexts\':\'all 15 x two labels retained; first round only FOUR exploratory contexts; never remove an unfavorable context\',\n \'Scenario_B\':\'BLOCKED until replicate provenance established; new development-only genes required per direction\',\n \'M3_final_fit\':\'new jointly tuned M2 final A-only fit supplies effective weights; never treat Step5 sequential fit as Task2 official M2\',\n \'M3_development\':\'permute that phase\\\'s frozen M2 weights; no permutation/seed selected using V or B\',\n \'M4_selection\':\'each randomized graph gets the identical development-only M1 coarse/fine search rule and budget\',\n \'metacells\':\'target aggregation, if used, follows input-only fixed assignment; final primary remains declared Step3 cell-gene scope\',\n \'conditional_EDF\':\'log-normalized expression-space tr(H)=sum_ij 1/(1+alpha*lambda_c_i+beta*gamma_g_j); cell representation and all weights held fixed. For metacell lifting with normalized membership columns, same trace applies to cell-level linear smoother. It is NOT adaptive total nonlinear degrees of freedom.\',\n \'randomization_failures\':\'save failed seed and diagnostic reason; do not silently replace/drop it or treat unchanged original graph as successful null\',\n \'random_M4_limitations\':\'unweighted degrees and raw weight multiset preserved; weighted degrees/connectivity not enforced; finite swap count does not establish uniform/mixed sampling\',\n \'previous_target_history\':history}\n for name,obj in [(\'method_registry.json\',methods),(\'common_protocol.json\',common),(\'parameter_grid_coarse.json\',grids),(\'randomization_policy.json\',{\'M3\':\'permutation of effective weights incl zeros; keep inherited normalization\',\'M4\':\'degree-preserving simple undirected double-edge swaps, raw weight permutation, per-base spectral normalization\',\'swap_multiplier\':c[\'swap_multiplier\'],\'max_attempts_per_swap\':c[\'max_attempts_per_swap\'],\'initial_seeds\':20,\'expanded_seeds\':100,\'all_seeds_reported\':True}),(\'previous_target_history.json\',history)]:\n  freeze(out/name,obj);freeze(configdir/name,obj)\n # JSON is a YAML 1.2-compatible representation; no PyYAML dependency needed.\n yaml_path=configdir/\'parameter_grid.yaml\';yaml_text=json.dumps(grids,ensure_ascii=False,indent=2,allow_nan=False)+\'\\n\'\n if yaml_path.exists() and yaml_path.read_text()!=yaml_text:raise ValueError(\'Frozen coarse parameter_grid.yaml differs\')\n yaml_path.write_text(yaml_text)\n root_yaml=root/\'configs/parameter_grid.yaml\'\n if root_yaml.exists() and root_yaml.read_text()!=yaml_text:raise ValueError(\'configs/parameter_grid.yaml already exists with another rule; retain it and use an explicit versioned grid workflow\')\n root_yaml.write_text(yaml_text)\n plan=seed_plan(c);plan.to_csv(out/\'random_seed_plan.csv\',index=False)\n method_table=pd.DataFrame(methods);method_table.to_csv(out/\'method_table.csv\',index=False)\n roster=pd.MultiIndex.from_product([LINES,DRUGS,REPS],names=[\'cell_line\',\'drug\',\'replicate\'])\n cover=cells.loc[cells.evaluation_eligible].groupby([\'cell_line\',\'drug\',\'replicate\'],observed=True).size().reindex(roster,fill_value=0).rename(\'n_cells\').reset_index();cover[\'n_pairs\']=cover.n_cells*len(genes)\n cover[\'execution_plan\']=[\'first_four_exploratory\' if (a,b) in FOUR else \'planned_later_all_contexts\' for a,b in zip(cover.cell_line,cover.drug)];cover.to_csv(out/\'all_context_roster.csv\',index=False)\n directions=[]\n for line in LINES:\n  for drug in DRUGS:\n   for a,b in [(REPS[0],REPS[1]),(REPS[1],REPS[0])]:directions.append({\'cell_line\':line,\'drug\':drug,\'development_replicate\':a,\'evaluation_replicate\':b,\'status\':\'BLOCKED_PENDING_REPLICATE_PROVENANCE\'})\n pd.DataFrame(directions).to_csv(out/\'cross_replicate_directions.csv\',index=False)\n na=len(c[\'alpha_coarse\']);nb=len(c[\'beta_coarse\']);nr=len(c[\'lambda_threshold_quantiles\'])+2\n budgets=[{\'method\':\'M0\',\'nominal_coarse_candidates_per_scope\':na,\'random_repeats\':1}, {\'method\':\'M1\',\'nominal_coarse_candidates_per_scope\':na*nb,\'random_repeats\':1},\n {\'method\':\'M2\',\'nominal_coarse_candidates_per_scope\':na*(1+(nb-1)*nr),\'random_repeats\':1}, {\'method\':\'M3\',\'nominal_coarse_candidates_per_scope\':0,\'random_repeats\':20}, {\'method\':\'M4\',\'nominal_coarse_candidates_per_scope\':na*nb,\'random_repeats\':20}]\n pd.DataFrame(budgets).to_csv(out/\'search_budget.csv\',index=False)\n null_columns=[\'scenario\',\'scope_id\',\'cell_line\',\'drug\',\'replicate\',\'phase\',\'method\',\'seed_index\',\'status\',\'failure_reason\',\'gene_universe_sha256\',\'Lc_sha256\',\'common_I_sha256\',\'alpha\',\'beta\',\'lambda_value\',\'lambda_rule\',\'mean_poisson_deviance\',\'normalized_mse\',\'raw_support_edges\',\'positive_edges\',\'zero_edges\',\'weight_sum\',\'trace_Lg\',\'lambda_max\',\'conditional_trace_H\',\'conditional_trace_H_reference\',\'conditional_trace_H_delta\',\'conditional_trace_H_relative_percent\',\'weighted_degree_preserved\',\'requested_swaps\',\'accepted_swaps\',\'attempts\',\'edge_overlap_fraction\']\n pd.DataFrame(columns=null_columns).to_csv(out/\'null_distribution_template.csv\',index=False)\n write_environment(root,out)\n summary={\'status\':\'five_method_and_coarse_rule_contracts_fixed_NOT_BENCHMARKED\',\'protocol_path\':str(out),\'config_path\':str(configdir),\'methods\':[\'M0\',\'M1\',\'M2\',\'M3\',\'M4\'],\n \'n_outer_cells\':n,\'n_outer_genes\':len(genes),\'n_outer_pairs\':n*len(genes),\'initial_random_seeds\':20,\'expanded_random_seeds\':100,\'search_budgets\':budgets,\n \'target_history_acknowledged\':True,\'new_final_B_read\':False,\'new_models_fitted\':False,\'real_randomized_graphs_generated\':False,\'fine_grid_requires_step2_2\':True,\n \'previous_target_lock_sha256_before\':history[\'target_access_lock_sha256\'],\'previous_target_lock_sha256_after\':digest(lockpath)}\n assert summary[\'previous_target_lock_sha256_before\']==summary[\'previous_target_lock_sha256_after\']\n save(out/\'task2_step1_summary.json\',summary);save(root/\'results/raw\'/f"{c[\'protocol_name\']}_step1_summary.json",summary)\n save(out/\'output_manifest.json\',{\'files\':[{\'name\':p.name,\'sha256\':digest(p)} for p in sorted(out.iterdir()) if p.is_file() and p.name!=\'output_manifest.json\']})\n return out,summary\n\ndef matched_shrinkage_audit(cell_eigenvalues,edges,reference_weights,control_weights,n_genes,alpha,beta):\n e,w0=canonical_edges(edges,n_genes,reference_weights);_,w1=canonical_edges(edges,n_genes,control_weights)\n if not np.array_equal(np.sort(w0),np.sort(w1)):raise ValueError(\'Control does not match effective-weight multiset\')\n base=conditional_edf(cell_eigenvalues,graph_laplacian(e,w0,n_genes),alpha,beta)\n control=conditional_edf(cell_eigenvalues,graph_laplacian(e,w1,n_genes),alpha,beta)\n return {\'trace_Lg_reference\':float(2*w0.sum()),\'trace_Lg_control\':float(2*w1.sum()),\'conditional_trace_H_reference\':base,\'conditional_trace_H_control\':control,\n \'conditional_trace_H_delta\':control-base,\'conditional_trace_H_relative_percent\':None if base<=0 else 100*(control-base)/base,\n \'interpretation\':\'conditional log-space linear smoother, all learned weights/representation held fixed; report difference, never select or reject permutations to match EDF after looking at targets\'}\n'
module_path=ROOT/"src/randomized_controls.py"
module_path.parent.mkdir(parents=True,exist_ok=True)
new_bytes=SOURCE.encode("utf-8")
if module_path.exists() and module_path.read_bytes()!=new_bytes:
    old_sha=hashlib.sha256(module_path.read_bytes()).hexdigest()
    backup=module_path.with_name("randomized_controls."+old_sha[:12]+".backup.py")
    if not backup.exists():backup.write_bytes(module_path.read_bytes())
module_path.write_bytes(new_bytes)
spec=importlib.util.spec_from_file_location("method_contract",module_path)
mc=importlib.util.module_from_spec(spec)
sys.modules["method_contract"]=mc
spec.loader.exec_module(mc)
print("Task 2.1 Source Code SHA256：",mc.ENGINE_SHA)


Task 2.1 Source Code SHA256： 3712e462217762223b90f6533e1260fb4a1d13c603ec570d5e01f53bedcaa6bf


In [3]:
CONFIG=mc.default_config(ROOT)
CONFIG["run_path"]=str(ROOT/"data/splits/step2_q05_seed20261008")
CONFIG["development_name"]="inner_q05_seed20261008"
CONFIG["prior_experiment"]="step5_four_contexts"
CONFIG["protocol_name"]="task2_five_methods"
mc.validate_config(CONFIG)
print(json.dumps(CONFIG,ensure_ascii=False,indent=2))


{
  "project_root": "/home/featurize/work/biological_prior_validation",
  "run_path": "/home/featurize/work/biological_prior_validation/data/splits/step2_q05_seed20261008",
  "development_name": "inner_q05_seed20261008",
  "prior_experiment": "step5_four_contexts",
  "protocol_name": "task2_five_methods",
  "master_seed": 20261008,
  "alpha_coarse": [
    0.0,
    0.001,
    0.01,
    0.1,
    1.0,
    10.0,
    100.0
  ],
  "beta_coarse": [
    0.0,
    1e-06,
    1e-05,
    0.0001,
    0.001,
    0.01,
    0.1,
    1.0,
    10.0,
    100.0
  ],
  "lambda_threshold_quantiles": [
    0.25,
    0.5,
    0.75,
    0.9,
    0.97
  ],
  "lambda_high_multiplier": 3.0,
  "initial_random_seeds": 20,
  "expanded_random_seeds": 100,
  "swap_multiplier": 10,
  "max_attempts_per_swap": 100,
  "cg_rtol": 1e-08,
  "cg_maxiter": 1000,
  "adaptive_max_outer": 100,
  "adaptive_x_rtol": 1e-07,
  "adaptive_s_atol": 1e-06
}


In [4]:
TEST_SOURCE = "from pathlib import Path\nimport sys,json,tempfile,shutil\nimport numpy as np\nimport pandas as pd\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]/'src'))\nimport method_contract as m\n\ndef reject(f):\n try:f()\n except (ValueError,RuntimeError):return\n raise AssertionError('Expected rejection')\n\ndef graph_tests():\n checks=[]\n n=10;edges=np.array([[i,(i+1)%8] for i in range(8)],dtype=int);edges=m.canonical_edges(edges,n);w=np.linspace(.4,.95,len(edges))\n rng=m.rng_for(20261008,'M3',0,0,'outer');effective=w*np.array([0,1,.5,0,.2,1,.4,.8])\n e,new,a=m.matched_permutation(edges,effective,n,rng)\n assert np.array_equal(e,edges) and np.array_equal(np.sort(new),np.sort(effective))\n assert a['before']['zero_weight_edges']==a['after']['zero_weight_edges']==2\n assert np.isclose(a['before']['weight_sum'],a['after']['weight_sum']) and np.isclose(a['before']['trace_Lg'],a['after']['trace_Lg'])\n assert not a['renormalized']\n _,allzero,az=m.matched_permutation(edges,np.zeros(len(edges)),n,m.rng_for(1,'M3',0,0,'inner'))\n assert np.array_equal(allzero,np.zeros(len(edges))) and az['after']['lambda_max']==0\n _,same,_=m.matched_permutation(edges,effective,n,m.rng_for(20261008,'M3',0,0,'outer'));assert np.array_equal(same,new)\n checks.append('M3 exact effective weight multiset/zeros/sum/trace and support, deterministic replay, all-zero case')\n en,wn,a=m.rewired_control(edges,w,n,m.rng_for(20261008,'M4',0,0,'outer'),swap_multiplier=5)\n assert np.array_equal(a['raw_reference']['degree_sequence'],a['raw_rewired']['degree_sequence'])\n assert a['swaps']['accepted_swaps']==len(edges)*5\n assert np.isclose(a['normalization']['after']['lambda_max'],1,rtol=1e-9)\n assert not a['weighted_degree_constraint_enforced']\n assert a['raw_rewired']['degree_sequence'][-2:]==[0,0]\n en2,wn2,_=m.rewired_control(edges,w,n,m.rng_for(20261008,'M4',0,0,'outer'),swap_multiplier=5)\n assert np.array_equal(en,en2) and np.array_equal(wn,wn2)\n checks.append('M4 simple graph degree sequence and isolates, deterministic swaps/weights, own spectral normalization')\n complete=np.array([(i,j) for i in range(4) for j in range(i+1,4)],dtype=int)\n reject(lambda:m.double_edge_swaps(complete,4,m.rng_for(1,'M4',0,0,'outer'),2,20))\n reject(lambda:m.canonical_edges(np.array([[0,1],[1,0]]),3))\n reject(lambda:m.canonical_edges(np.array([[1,1]]),3))\n reject(lambda:m.normalize_base(edges,np.zeros(len(edges)),n))\n reject(lambda:m.graph_laplacian(edges,np.full(len(edges),np.nan),n))\n checks.append('complete-graph swap failure, duplicates/self loops/NaN and zero-base normalization rejected')\n L=m.graph_laplacian(edges,effective,n);lc=np.array([0.,.2,1.]);alpha=.3;beta=.7\n H=np.linalg.inv(np.eye(len(lc)*n)+alpha*np.kron(np.diag(lc),np.eye(n))+beta*np.kron(np.eye(len(lc)),L.toarray()))\n assert np.isclose(m.conditional_edf(lc,L,alpha,beta),np.trace(H),rtol=1e-12)\n assert np.isclose(m.conditional_edf(lc,L,0,0),len(lc)*n)\n assert np.isclose(m.conditional_edf(lc,L,alpha,0),n*np.sum(1/(1+alpha*lc)))\n assert m.conditional_edf(lc,L,alpha,beta)<=m.conditional_edf(lc,L,alpha,0)\n audit=m.matched_shrinkage_audit(lc,edges,effective,new,n,alpha,beta)\n assert np.isclose(audit['trace_Lg_reference'],audit['trace_Lg_control'])\n assert np.isclose(audit['conditional_trace_H_delta'],audit['conditional_trace_H_control']-audit['conditional_trace_H_reference'])\n checks.append('conditional tr(H) agrees with independent dense Kronecker oracle, alpha/beta-zero and shrinkage bounds')\n X=np.arange(20,dtype=float).reshape(2,10)/10;beta=.2;tau=.5;value=m.lambda_from_threshold(beta,tau)\n d2=np.sum((X[:,edges[:,0]]-X[:,edges[:,1]])**2,axis=0)\n assert np.allclose(m.adaptive_reliability(X,edges,beta,value),np.clip(1-d2/tau,0,1))\n assert np.array_equal(m.adaptive_reliability(X,edges,0,1),np.ones(len(edges)))\n assert np.array_equal(m.adaptive_reliability(X,edges,beta,'fixed_limit'),np.ones(len(edges)))\n reject(lambda:m.adaptive_reliability(X,edges,beta,0))\n checks.append('manuscript lambda <-> old tau identity, beta-zero and fixed limit')\n # Exercise large-graph eigsh path with a deterministic non-null starting vector.\n chain=np.array([[i,i+1] for i in range(299)],dtype=int)\n radius=m.graph_audit(chain,np.ones(299),300)['lambda_max']\n assert np.isclose(radius,2+2*np.cos(np.pi/300),rtol=1e-9)\n checks.append('large-graph spectral audit eigsh agrees with analytic path-graph maximum eigenvalue')\n return {'status':'PASS','checks':checks}\n\ndef synthetic_project(root):\n run=root/'data/splits/step2_fixture';prep=run/'A_preprocessing';dev=run/'development/inner_q05_seed20261008_v1';prior=run/'step5/step5_four_contexts_v1'\n for p in [prep,dev,prior,run/'evaluation',root/'data/processed']:p.mkdir(parents=True,exist_ok=True)\n cells=pd.DataFrame({'matrix_row':np.arange(31),'cell_barcode':[f'c{i}' for i in range(31)],'evaluation_eligible':[True]*30+[False],\n 'cell_line':pd.Series([l for l in m.LINES for d in m.DRUGS for r in m.REPS]+[pd.NA],dtype='string'),\n 'drug':pd.Series([d for l in m.LINES for d in m.DRUGS for r in m.REPS]+[pd.NA],dtype='string'),\n 'replicate':pd.Series([r for l in m.LINES for d in m.DRUGS for r in m.REPS]+[pd.NA],dtype='string')})\n genes=pd.DataFrame({'selected_col':[0,1,2],'gene_idx':[1,2,3],'gene_symbol':['a','b','c']});cp=prep/'evaluation_cells.parquet';gp=prep/'selected_genes_A.parquet';cells.to_parquet(cp,index=False);genes.to_parquet(gp,index=False)\n pair={'cells_file':cp.name,'genes_file':gp.name,'cells_sha256':m.digest(cp),'genes_sha256':m.digest(gp),'n_cells':30,'n_genes':3,'n_pairs':90,'q':.5,'normalization_scale':10000}\n m.save(prep/'evaluation_pair_contract.json',pair);metric={**pair,'pair_contract_sha256':m.digest(prep/'evaluation_pair_contract.json'),'mu_floor':1e-8,'relative_min_denominator':1e-12,'primary':'mean Poisson deviance','secondary':'normalized MSE'};m.save(run/'evaluation/metric_contract.json',metric);m.save(run/'identity.json',{'seed':20261008})\n cells.to_parquet(dev/'cell_roles.parquet',index=False);inner=[]\n for r in m.REPS:\n  p=dev/f'{r}_inner_selected_genes.parquet';genes.to_parquet(p,index=False);inner.append({'replicate':r,'selected_genes_sha256':m.digest(p)})\n m.save(dev/'step4_summary.json',{'status':'step4_prepared','metric_contract_sha256_after':m.digest(run/'evaluation/metric_contract.json'),'inner_genes_by_replicate':inner})\n m.save(dev/'scenario_A_contract.json',{'inner_q':.5});m.save(dev/'identity.json',{'status':'frozen'})\n m.save(prior/'step5_summary.json',{'status':'four_context_baseline_scored','final_B_accessed':True,'contexts':m.FOUR});m.save(prior/'frozen_model_registry.json',{'status':'frozen'});m.save(run/'evaluation/target_access_lock.json',{'status':'already_accessed_keep_me'})\n pd.DataFrame({'gene_a':['a','b'],'gene_b':['b','c'],'ppi_weight':[.7,.8]}).to_parquet(root/'data/processed/ppi_annotation_reference_edges.parquet',index=False)\n # Poison target and old-score files: preparation must not read them.\n (run/'B_target.npz').write_bytes(b'POISON');(prior/'baseline_reproduction.csv').write_bytes(b'POISON')\n c=m.default_config(root);c['run_path']=str(run);return c\n\ndef integration_tests():\n with tempfile.TemporaryDirectory() as t:\n  c=synthetic_project(Path(t));lock=Path(c['run_path'])/'evaluation/target_access_lock.json';before=m.digest(lock)\n  out,s=m.prepare_contract(c);assert not s['new_final_B_read'] and not s['new_models_fitted']\n  assert m.digest(lock)==before\n  plan=pd.read_csv(out/'random_seed_plan.csv');assert len(plan)==30*2*2*100 and int(plan.initial_round.sum())==30*2*2*20\n  assert plan.entropy_json.is_unique\n  assert len(pd.read_csv(out/'all_context_roster.csv'))==30\n  assert len(pd.read_csv(out/'cross_replicate_directions.csv'))==30\n  assert pd.read_csv(out/'cross_replicate_directions.csv').status.eq('BLOCKED_PENDING_REPLICATE_PROVENANCE').all()\n  b=pd.read_csv(out/'search_budget.csv').set_index('method').nominal_coarse_candidates_per_scope\n  assert b.M1==b.M4 and b.M0==7 and b.M1==70 and b.M2==448 and b.M3==0\n  assert len(pd.read_csv(out/'null_distribution_template.csv'))==0\n  out2,s2=m.prepare_contract(c);assert s==s2\n  reject(lambda:m.prepare_contract({**c,'beta_coarse':[0.,1.]}))\n  reject(lambda:m.prepare_contract({**c,'initial_random_seeds':19}))\n  assert m.digest(lock)==before\n return {'status':'PASS','checks':['whole protocol preparation including NA metadata and previously read B history','M1/M4 equal joint budgets; M2 lambda Cartesian search','20/100 seed plan nested, unique streams and all contexts/directions retained','no benchmark results fabricated; no target/old-score files read','immutable retry and changed rules rejected; prior lock unchanged']}\n\nif __name__=='__main__':print(json.dumps({'graph_tests':graph_tests(),'integration_tests':integration_tests()},indent=2))\n"
test_namespace={"__name__":"task2_embedded_tests","__file__":str(ROOT/"tests/test_method_contract.py")}
exec(compile(TEST_SOURCE,"task2_embedded_tests.py","exec"),test_namespace)
TEST_REPORT={"graph_tests":test_namespace["graph_tests"](),"integration_tests":test_namespace["integration_tests"]()}
assert all(x["status"]=="PASS" for x in TEST_REPORT.values())
print(json.dumps(TEST_REPORT,ensure_ascii=False,indent=2))


{
  "graph_tests": {
    "status": "PASS",
    "checks": [
      "M3 exact effective weight multiset/zeros/sum/trace and support, deterministic replay, all-zero case",
      "M4 simple graph degree sequence and isolates, deterministic swaps/weights, own spectral normalization",
      "complete-graph swap failure, duplicates/self loops/NaN and zero-base normalization rejected",
      "conditional tr(H) agrees with independent dense Kronecker oracle, alpha/beta-zero and shrinkage bounds",
      "manuscript lambda <-> old tau identity, beta-zero and fixed limit",
      "large-graph spectral audit eigsh agrees with analytic path-graph maximum eigenvalue"
    ]
  },
  "integration_tests": {
    "status": "PASS",
    "checks": [
      "whole protocol preparation including NA metadata and previously read B history",
      "M1/M4 equal joint budgets; M2 lambda Cartesian search",
      "20/100 seed plan nested, unique streams and all contexts/directions retained",
      "no benchmark results fa

In [5]:
assert all(x["status"]=="PASS" for x in TEST_REPORT.values())
PROTOCOL_PATH,SUMMARY=mc.prepare_contract(CONFIG)
mc.save(PROTOCOL_PATH/"synthetic_test_report.json",TEST_REPORT)
manifest=mc.js(PROTOCOL_PATH/"output_manifest.json")
manifest["files"]=[x for x in manifest["files"] if x["name"]!="synthetic_test_report.json"]
manifest["files"].append({"name":"synthetic_test_report.json","sha256":mc.digest(PROTOCOL_PATH/"synthetic_test_report.json")})
mc.save(PROTOCOL_PATH/"output_manifest.json",manifest)
print(json.dumps(SUMMARY,ensure_ascii=False,indent=2))


{
  "status": "five_method_and_coarse_rule_contracts_fixed_NOT_BENCHMARKED",
  "protocol_path": "/home/featurize/work/biological_prior_validation/data/splits/step2_q05_seed20261008/task2_protocols/task2_five_methods",
  "config_path": "/home/featurize/work/biological_prior_validation/configs/task2_five_methods",
  "methods": [
    "M0",
    "M1",
    "M2",
    "M3",
    "M4"
  ],
  "n_outer_cells": 989299,
  "n_outer_genes": 2500,
  "n_outer_pairs": 2473247500,
  "initial_random_seeds": 20,
  "expanded_random_seeds": 100,
  "search_budgets": [
    {
      "method": "M0",
      "nominal_coarse_candidates_per_scope": 7,
      "random_repeats": 1
    },
    {
      "method": "M1",
      "nominal_coarse_candidates_per_scope": 70,
      "random_repeats": 1
    },
    {
      "method": "M2",
      "nominal_coarse_candidates_per_scope": 448,
      "random_repeats": 1
    },
    {
      "method": "M3",
      "nominal_coarse_candidates_per_scope": 0,
      "random_repeats": 20
    },
    {
    

In [6]:
methods=pd.read_csv(PROTOCOL_PATH/"method_table.csv")
budget=pd.read_csv(PROTOCOL_PATH/"search_budget.csv")
roster=pd.read_csv(PROTOCOL_PATH/"all_context_roster.csv")
seeds=pd.read_csv(PROTOCOL_PATH/"random_seed_plan.csv")
assert int(roster.n_pairs.sum())==SUMMARY["n_outer_pairs"]
assert seeds.entropy_json.is_unique
assert budget.set_index("method").loc["M1","nominal_coarse_candidates_per_scope"]==budget.set_index("method").loc["M4","nominal_coarse_candidates_per_scope"]
print(methods[["method","name","optimization"]].to_string(index=False))
print(budget.to_string(index=False))
print(roster.to_string(index=False))
print("Seed slots (planned, not executed):",len(seeds))
print("Initial-round slots:",int(seeds.initial_round.sum()))
print("Cross-replicate status:")
print(pd.read_csv(PROTOCOL_PATH/"cross_replicate_directions.csv").head().to_string(index=False))


method                     name                                                                             optimization
    M0                Cell-only                                                                       alpha only; beta=0
    M1                Fixed PPI                                           joint Cartesian alpha x beta, including beta=0
    M2             Adaptive PPI joint alpha x beta x lambda-threshold-rule; includes fixed-PPI infinity limit and beta=0
    M3 Matched random weighting                         inherit selected M2 alpha/beta/lambda; no per-permutation tuning
    M4           Randomized PPI                   independently joint alpha x beta per seed, same grid/rule/budget as M1
method  nominal_coarse_candidates_per_scope  random_repeats
    M0                                    7               1
    M1                                   70               1
    M2                                  448               1
    M3                            

In [7]:
assert SUMMARY["new_final_B_read"] is False
assert SUMMARY["new_models_fitted"] is False
assert SUMMARY["real_randomized_graphs_generated"] is False
assert SUMMARY["previous_target_lock_sha256_before"]==SUMMARY["previous_target_lock_sha256_after"]
REVIEW_ZIP=ROOT/"results/summary/task2_step1_review_bundle.zip"
REVIEW_ZIP.parent.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(REVIEW_ZIP,"w",zipfile.ZIP_DEFLATED) as z:
    for p in sorted(PROTOCOL_PATH.iterdir()):
        if p.is_file():z.write(p,p.name)
    for p in sorted((ROOT/"configs"/CONFIG["protocol_name"]).iterdir()):
        if p.is_file():z.write(p,"configs/"+p.name)
print("2.1 Method rules are prepared. Please return:")
print(REVIEW_ZIP)
print("Next step: 2.2 Fix the complete parameter grid and the same-budget coarse-to-fine search rule.")


2.1 Method rules are prepared. Please return:
/home/featurize/work/biological_prior_validation/results/summary/task2_step1_review_bundle.zip
Next step: 2.2 Fix the complete parameter grid and the same-budget coarse-to-fine search rule.
